# 01 Exploratory Data Analysis

Understanding the raw Telco dataset before any processing.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

df = pd.read_csv('../data/raw/telco_churn.csv')
print(df.shape)
df.head()

## Schema & dtypes

In [ ]:
df.info()

## Missing values

`TotalCharges` is read as object - some rows have a space string instead of a number.

In [ ]:
# Coerce to numeric to expose hidden nulls
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

missing = df.isnull().sum()
print(missing[missing > 0])

In [ ]:
# Who are the null rows?
df[df['TotalCharges'].isnull()][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']]

All 11 null rows have `tenure = 0` — new customers with no charges yet. Median imputation is safe.

## Target distribution

In [ ]:
churn_counts = df['Churn'].value_counts()
print(churn_counts)
print(f"\nChurn rate: {(df['Churn'] == 'Yes').mean()*100:.1f}%")

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(churn_counts.index, churn_counts.values, color=['steelblue', 'tomato'])
ax.set_title('Churn Distribution')
ax.set_ylabel('Customers')
for i, v in enumerate(churn_counts.values):
    ax.text(i, v + 30, str(v), ha='center')
plt.tight_layout()
plt.show()

26.5% churn — imbalanced. Will need SMOTE or class weighting at training time.

## Numerical feature distributions

In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for ax, col in zip(axes, num_cols):
    churned     = df[df['Churn'] == 'Yes'][col].dropna()
    not_churned = df[df['Churn'] == 'No'][col].dropna()
    ax.hist(not_churned, bins=30, alpha=0.6, label='Retained', color='steelblue')
    ax.hist(churned,     bins=30, alpha=0.6, label='Churned',  color='tomato')
    ax.set_title(col)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Churn rate by categorical features

In [ ]:
cat_cols = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents',
    'PhoneService', 'InternetService', 'Contract',
    'PaymentMethod', 'TechSupport', 'OnlineSecurity',
]

df['churn_bin'] = (df['Churn'] == 'Yes').astype(int)

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, col in zip(axes.flat, cat_cols):
    rates = df.groupby(col)['churn_bin'].mean().sort_values(ascending=False)
    ax.bar(rates.index.astype(str), rates.values * 100, color='steelblue')
    ax.set_title(col, fontsize=9)
    ax.set_ylabel('Churn %')
    ax.tick_params(axis='x', labelsize=7)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))

plt.suptitle('Churn Rate by Categorical Feature', y=1.02, fontsize=11)
plt.tight_layout()
plt.show()

## Correlation — numerical features vs churn

In [ ]:
corr = df[num_cols + ['churn_bin']].corr()['churn_bin'].drop('churn_bin').sort_values()

fig, ax = plt.subplots(figsize=(5, 3))
ax.barh(corr.index, corr.values, color=['tomato' if v > 0 else 'steelblue' for v in corr.values])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Correlation with Churn')
plt.tight_layout()
plt.show()

print(corr)

## Key observations

- **Tenure**: churners skew heavily toward low tenure (new customers)
- **MonthlyCharges**: churners pay more per month on average
- **Contract**: month-to-month customers churn far more than annual/two-year
- **TechSupport / OnlineSecurity**: no-service customers churn more
- Class imbalance (~27% churn) needs to be handled before training